# 04 — Filtering e Hybrid Search no Qdrant

Uma das vantagens do Qdrant sobre competidores e o suporte nativo a **filtros de payload combinados com busca vetorial** sem perda de performance.

## O que vamos aprender

1. Filtros por campos de payload (must, should, must_not)
2. Indexar campos de payload para busca rapida
3. Busca hibrida: dense vectors + sparse (BM25)
4. Score fusion: RRF e score ponderado

In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, SparseVectorParams, PointStruct,
    Filter, FieldCondition, MatchValue, MatchAny, Range,
    PayloadSchemaType, SparseVector,
    VectorsConfig,
)
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import numpy as np
import pandas as pd

client = QdrantClient(host='localhost', port=6333)
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Pronto!')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Pronto!


In [2]:
# Dataset rico para demonstrar filtros
artigos = [
    {'id': 1, 'titulo': 'Introducao ao Python', 'categoria': 'programacao', 'nivel': 'basico', 'ano': 2020, 'rating': 4.5},
    {'id': 2, 'titulo': 'Machine Learning com PyTorch', 'categoria': 'ml', 'nivel': 'avancado', 'ano': 2023, 'rating': 4.8},
    {'id': 3, 'titulo': 'SQL para Analistas', 'categoria': 'banco_de_dados', 'nivel': 'intermediario', 'ano': 2021, 'rating': 4.2},
    {'id': 4, 'titulo': 'Docker e Kubernetes', 'categoria': 'devops', 'nivel': 'intermediario', 'ano': 2022, 'rating': 4.6},
    {'id': 5, 'titulo': 'Deep Learning com Transformers', 'categoria': 'ml', 'nivel': 'avancado', 'ano': 2023, 'rating': 4.9},
    {'id': 6, 'titulo': 'APIs REST com FastAPI', 'categoria': 'programacao', 'nivel': 'intermediario', 'ano': 2022, 'rating': 4.7},
    {'id': 7, 'titulo': 'Qdrant para Buscas Semanticas', 'categoria': 'banco_de_dados', 'nivel': 'avancado', 'ano': 2023, 'rating': 4.8},
    {'id': 8, 'titulo': 'Embeddings e RAG na Pratica', 'categoria': 'ml', 'nivel': 'intermediario', 'ano': 2024, 'rating': 5.0},
    {'id': 9, 'titulo': 'Seguranca em APIs Web', 'categoria': 'seguranca', 'nivel': 'avancado', 'ano': 2023, 'rating': 4.4},
    {'id': 10, 'titulo': 'Python para Iniciantes', 'categoria': 'programacao', 'nivel': 'basico', 'ano': 2019, 'rating': 4.1},
]

titulos = [a['titulo'] for a in artigos]
embs = model.encode(titulos, normalize_embeddings=True)

if client.collection_exists('artigos'):
    client.delete_collection('artigos')
client.create_collection(
    collection_name='artigos',
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

points = [
    PointStruct(
        id=a['id'],
        vector=embs[i].tolist(),
        payload={k: v for k, v in a.items() if k != 'id'},
    )
    for i, a in enumerate(artigos)
]
client.upsert('artigos', points=points)

# Criar indices nos campos de payload para busca rapida
for campo in ['categoria', 'nivel', 'ano', 'rating']:
    client.create_payload_index(
        collection_name='artigos',
        field_name=campo,
        field_schema=PayloadSchemaType.KEYWORD if campo in ('categoria', 'nivel') else PayloadSchemaType.FLOAT,
    )

print(f'{len(artigos)} artigos indexados com payload indexes')

10 artigos indexados com payload indexes


## 4.1 Filtros de Payload

In [3]:
query_vec = model.encode('aprendizagem de maquina e redes neurais', normalize_embeddings=True)

def buscar_com_filtro(query_vec, filtro, descricao):
    results = client.query_points(
        collection_name='artigos',
        query=query_vec.tolist(),
        limit=5,
        query_filter=filtro,
        with_payload=True,
    ).points
    print(f'\n{descricao}:')
    for r in results:
        p = r.payload
        print(f'  {r.score:.3f} | {p["titulo"]} ({p["nivel"]}, {p["ano"]}, rating={p["rating"]})')

# Filtro 1: Apenas ML avancado
buscar_com_filtro(
    query_vec,
    Filter(must=[
        FieldCondition(key='categoria', match=MatchValue(value='ml')),
        FieldCondition(key='nivel', match=MatchValue(value='avancado')),
    ]),
    'ML + Avancado'
)

# Filtro 2: Qualquer categoria, mas rating >= 4.7
buscar_com_filtro(
    query_vec,
    Filter(must=[
        FieldCondition(key='rating', range=Range(gte=4.7))
    ]),
    'Rating >= 4.7'
)

# Filtro 3: ML ou Banco de dados (should = OR)
buscar_com_filtro(
    query_vec,
    Filter(should=[
        FieldCondition(key='categoria', match=MatchValue(value='ml')),
        FieldCondition(key='categoria', match=MatchValue(value='banco_de_dados')),
    ]),
    'ML ou Banco de Dados'
)

# Filtro 4: Qualquer exceto programacao basico
buscar_com_filtro(
    query_vec,
    Filter(must_not=[
        FieldCondition(key='nivel', match=MatchValue(value='basico')),
    ]),
    'Excluindo nivel basico'
)


ML + Avancado:
  0.010 | Machine Learning com PyTorch (avancado, 2023, rating=4.8)
  -0.038 | Deep Learning com Transformers (avancado, 2023, rating=4.9)



Rating >= 4.7:
  0.402 | Embeddings e RAG na Pratica (intermediario, 2024, rating=5.0)
  0.217 | Qdrant para Buscas Semanticas (avancado, 2023, rating=4.8)
  0.072 | APIs REST com FastAPI (intermediario, 2022, rating=4.7)
  0.010 | Machine Learning com PyTorch (avancado, 2023, rating=4.8)
  -0.038 | Deep Learning com Transformers (avancado, 2023, rating=4.9)



ML ou Banco de Dados:
  0.402 | Embeddings e RAG na Pratica (intermediario, 2024, rating=5.0)
  0.217 | Qdrant para Buscas Semanticas (avancado, 2023, rating=4.8)
  0.097 | SQL para Analistas (intermediario, 2021, rating=4.2)
  0.010 | Machine Learning com PyTorch (avancado, 2023, rating=4.8)
  -0.038 | Deep Learning com Transformers (avancado, 2023, rating=4.9)



Excluindo nivel basico:
  0.402 | Embeddings e RAG na Pratica (intermediario, 2024, rating=5.0)
  0.217 | Qdrant para Buscas Semanticas (avancado, 2023, rating=4.8)
  0.182 | Seguranca em APIs Web (avancado, 2023, rating=4.4)
  0.133 | Docker e Kubernetes (intermediario, 2022, rating=4.6)
  0.097 | SQL para Analistas (intermediario, 2021, rating=4.2)


## 4.2 Hybrid Search: Dense + Sparse (BM25)

**Dense vectors** (semantica) + **Sparse vectors** (keywords) = melhor retrieval em geral.

Implementamos manualmente com RRF (Reciprocal Rank Fusion).

In [4]:
# BM25 para sparse retrieval
corpus_tokenizado = [t.lower().split() for t in titulos]
bm25 = BM25Okapi(corpus_tokenizado)

def hybrid_search_rrf(query, top_k=5, alpha=0.5):
    """
    Hybrid search com Reciprocal Rank Fusion (RRF).
    alpha = peso do dense (0=apenas BM25, 1=apenas dense)
    """
    # Dense search
    q_vec = model.encode(query, normalize_embeddings=True)
    dense_results = client.query_points(
        collection_name='artigos',
        query=q_vec.tolist(),
        limit=len(artigos),
        with_payload=True,
    ).points
    dense_ranks = {r.id: i + 1 for i, r in enumerate(dense_results)}
    dense_scores = {r.id: r.score for r in dense_results}
    
    # Sparse search (BM25)
    q_tokens = query.lower().split()
    bm25_scores = bm25.get_scores(q_tokens)
    bm25_ranks = {artigos[i]['id']: rank + 1 
                  for rank, i in enumerate(np.argsort(bm25_scores)[::-1])}
    bm25_scores_dict = {artigos[i]['id']: bm25_scores[i] for i in range(len(artigos))}
    
    # RRF fusion
    k = 60  # constante RRF
    all_ids = set(dense_ranks.keys()) | set(bm25_ranks.keys())
    
    fused_scores = {}
    for doc_id in all_ids:
        dense_rrf = 1.0 / (k + dense_ranks.get(doc_id, len(artigos) + k))
        bm25_rrf = 1.0 / (k + bm25_ranks.get(doc_id, len(artigos) + k))
        fused_scores[doc_id] = alpha * dense_rrf + (1 - alpha) * bm25_rrf
    
    top_ids = sorted(fused_scores, key=fused_scores.get, reverse=True)[:top_k]
    
    return [
        {
            'id': doc_id,
            'titulo': next(a['titulo'] for a in artigos if a['id'] == doc_id),
            'fused_score': fused_scores[doc_id],
            'dense_score': dense_scores.get(doc_id, 0),
            'bm25_score': bm25_scores_dict.get(doc_id, 0),
        }
        for doc_id in top_ids
    ]

# Comparar dense vs sparse vs hybrid
queries_teste = [
    'aprendizado profundo com transformers',  # semantico — favorece dense
    'Python basico iniciantes',               # keywords exatas — favorece BM25
    'Qdrant busca semantica banco',           # hibrido
]

for query in queries_teste:
    print(f'\nQuery: "{query}"')
    print('-' * 50)
    results = hybrid_search_rrf(query, top_k=3, alpha=0.5)
    for r in results:
        print(f'  Fused:{r["fused_score"]:.4f} | Dense:{r["dense_score"]:.3f} | BM25:{r["bm25_score"]:.2f} | {r["titulo"]}')


Query: "aprendizado profundo com transformers"
--------------------------------------------------


  Fused:0.0164 | Dense:0.532 | BM25:2.52 | Deep Learning com Transformers
  Fused:0.0157 | Dense:0.304 | BM25:0.00 | Python para Iniciantes
  Fused:0.0154 | Dense:0.254 | BM25:0.00 | Seguranca em APIs Web

Query: "Python basico iniciantes"
--------------------------------------------------


  Fused:0.0164 | Dense:0.804 | BM25:3.36 | Python para Iniciantes
  Fused:0.0161 | Dense:0.689 | BM25:1.34 | Introducao ao Python
  Fused:0.0156 | Dense:0.156 | BM25:0.00 | Seguranca em APIs Web

Query: "Qdrant busca semantica banco"
--------------------------------------------------


  Fused:0.0164 | Dense:0.794 | BM25:1.78 | Qdrant para Buscas Semanticas
  Fused:0.0160 | Dense:0.338 | BM25:0.00 | Seguranca em APIs Web
  Fused:0.0155 | Dense:0.212 | BM25:0.00 | Embeddings e RAG na Pratica


## Resumo

| Tecnica | Quando usar | Implementacao |
|---------|------------|---------------|
| Apenas dense | Queries semanticas gerais | `client.search()` |
| Filtros payload | Restringir por metadados | `search(query_filter=...)` |
| BM25 + Dense (RRF) | Queries com keywords especificas | Dois indices + fusion |
| Qdrant nativo sparse | Dense + sparse num unico query | `NamedSparseVector` |

**Para RAG de producao:** Hybrid search com RRF normalmente supera dense-only em 5-15% de recall.